📝 Summary of Components
Model: Flan-T5 Small (google/flan-t5-small)

New Task: SST-2 sentiment classification (GLUE benchmark)

LwF: Combines cross-entropy for new task + KL divergence between M_old and M_new outputs

Distillation: Uses softened logits with temperature scaling

Output: Shows model predictions before and after fine-tuning



In [ ]:
# Upgrade datasets and fsspec
!pip install --upgrade datasets fsspec

# Restart the notebook kernel after upgrading the libraries
# (This is important for the changes to take effect)

# Then, re-run your code from the beginning.

import torch
from torch.utils.data import DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration
from torch.optim import AdamW
from datasets import load_dataset, disable_caching
import torch.nn.functional as F
import os
import shutil

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Added: Clear dataset cache ---
# Identify the default cache directory
datasets_cache_dir = os.path.expanduser("~/.cache/huggingface/datasets")

# Check if the directory exists and remove it
if os.path.exists(datasets_cache_dir):
    print(f"Clearing dataset cache at: {datasets_cache_dir}")
    shutil.rmtree(datasets_cache_dir)
    print("Cache cleared.")
else:
    print(f"Dataset cache directory not found at: {datasets_cache_dir}")

# Optional: Temporarily disable caching for the next load operation
# disable_caching()
# --- End Added ---


# Load tokenizer and models
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-small")
model_new = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small").to(device)
model_old = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small").to(device)
model_old.eval()
for param in model_old.parameters():
    param.requires_grad = False

# Load dataset (GLUE SST-2 for sentiment classification)
# Added instructions to clear cache above
dataset = load_dataset("glue", "sst2") # This line should now work with updated libraries and cleared cache
train_data = dataset["train"].select(range(1000))   # small subset for training
val_data = dataset["validation"].select(range(100)) # small subset for eval

# Preprocessing
def preprocess(example):
    input_text = f"sst2 sentence: {example['sentence']}"
    target_text = "positive" if example["label"] == 1 else "negative"
    inputs = tokenizer(input_text, padding="max_length", max_length=64, truncation=True, return_tensors="pt")
    targets = tokenizer(target_text, padding="max_length", max_length=8, truncation=True, return_tensors="pt")
    return {
        "input_ids": inputs.input_ids.squeeze(0),
        "attention_mask": inputs.attention_mask.squeeze(0),
        "labels": targets.input_ids.squeeze(0)
    }

train_dataset = [preprocess(ex) for ex in train_data]
val_dataset = [preprocess(ex) for ex in val_data]
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

# Loss function for LwF
def lwf_loss(old_logits, new_logits, temperature=2.0):
    p_old = F.softmax(old_logits / temperature, dim=-1)
    p_new = F.log_softmax(new_logits / temperature, dim=-1)
    return F.kl_div(p_new, p_old, reduction='batchmean') * (temperature ** 2)

# Training setup
optimizer = AdamW(model_new.parameters(), lr=5e-5)
lambda_new = 1.0
lambda_lwf = 0.5
temperature = 2.0
epochs = 3

# Training loop
model_new.train()
for epoch in range(epochs):
    total_loss = 0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # New model forward
        outputs_new = model_new(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss_ce = outputs_new.loss

        # Distillation from old model
        with torch.no_grad():
            logits_old = model_old(input_ids=input_ids, attention_mask=attention_mask).logits

        logits_new = model_new(input_ids=input_ids, attention_mask=attention_mask).logits
        loss_distill = lwf_loss(logits_old, logits_new, temperature)

        # Combine losses
        loss = lambda_new * loss_ce + lambda_lwf * loss_distill
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}/{epochs} | Total Loss: {total_loss:.4f}")

# Evaluation: Compare outputs
model_new.eval()
sample = val_dataset[0]
input_ids = sample["input_ids"].unsqueeze(0).to(device)
attention_mask = sample["attention_mask"].unsqueeze(0).to(device)

with torch.no_grad():
    old_output = model_old.generate(input_ids=input_ids, attention_mask=attention_mask)
    new_output = model_new.generate(input_ids=input_ids, attention_mask=attention_mask)

print("\n=== Sample Comparison ===")
print("Input:        ", tokenizer.decode(input_ids[0], skip_special_tokens=True))
print("Old Output:   ", tokenizer.decode(old_output[0], skip_special_tokens=True))
print("New Output:   ", tokenizer.decode(new_output[0], skip_special_tokens=True))